# Dockerfile and Custom Images


## Learning Goals

By the end of this notebook, you should be able to:

- explain what a Dockerfile is
- build a custom image for a simple Python project
- run a container from your image
- understand `FROM`, `WORKDIR`, `COPY`, `RUN`, `CMD`, and `EXPOSE`
- use `.dockerignore`
- understand when ports, logs, and volumes are needed

We use simple Python examples first. Django/DRF Dockerfiles will be covered in a later deployment lesson.


## What is a Dockerfile?

A **Dockerfile** is a text file that describes how to build a Docker image.

It answers questions like:

- Which base image should we start from?
- Which files should be copied into the image?
- Which packages should be installed?
- What command should run when the container starts?

Simple flow:

```text
Dockerfile + project files --docker build--> image --docker run--> container
```


## Build Time vs Run Time

Two phases are important:

| Phase | Command | What happens |
|-------|---------|--------------|
| Build time | `docker build` | Docker creates an image from the Dockerfile. |
| Run time | `docker run` | Docker starts a container from an image. |

Example:

```bash
docker build -t my_python_app .
docker run my_python_app
```

A common beginner mistake is confusing build-time commands with run-time commands.


## Important Dockerfile Commands

| Command | Purpose |
|---------|---------|
| `FROM` | Choose a base image. |
| `WORKDIR` | Set working directory inside the image/container. |
| `COPY` | Copy files from host into image. |
| `RUN` | Execute commands while building the image. |
| `CMD` | Default command when a container starts. |
| `ENTRYPOINT` | Fixed executable for the container. |
| `EXPOSE` | Document which port a network app uses. |

Important difference:

```text
RUN = happens while building the image
CMD = happens when running the container
```


## Sample Projects

The samples are in:

```text
session45/samples/
```

| Sample | Main concept |
|--------|--------------|
| `greeting_app` | `CMD` and command override |
| `config_reader` | copying multiple files into an image |
| `report_writer` | writing files and why volumes matter |
| `looping_worker` | long-running containers, detached mode, logs, stop |
| `simple_http` | request/response and port mapping |

This order is intentional:

```text
run command → copy files → persist data → long-running process → expose port
```


## Sample 1: Greeting App

Files:

```text
session45/samples/greeting_app/
    Dockerfile
    greet.py
```

Dockerfile:

```dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY greet.py /app/

CMD ["python", "greet.py"]
```

Build and run:

```bash
cd session45/samples/greeting_app
docker build -t greeting_app .
docker run greeting_app
```

Override the default command:

```bash
docker run greeting_app python greet.py Sara
```

Concept:

> `CMD` is the default command, but you can override it when running the container.


## Sample 2: Config Reader

Files:

```text
session45/samples/config_reader/
    Dockerfile
    config.json
    read_config.py
```

This sample reads values from `config.json`.

Dockerfile:

```dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY read_config.py config.json /app/

CMD ["python", "read_config.py"]
```

Build and run:

```bash
cd session45/samples/config_reader
docker build -t config_reader .
docker run config_reader
```

Concept:

> Files must be copied into the image if the container should use them.


## Sample 3: Report Writer

Files:

```text
session45/samples/report_writer/
    Dockerfile
    write_report.py
```

This sample writes a file:

```text
output/report.txt
```

Build and run:

```bash
cd session45/samples/report_writer
docker build -t report_writer .
docker run report_writer
```

But files written only inside a container are not a good place for important data.

Use a bind mount to keep output on your computer:

```bash
mkdir -p output
docker run -v ./output:/app/output report_writer
```

Concept:

> Containers are temporary. Use volumes or bind mounts for data that must survive.


## Sample 4: Looping Worker

Files:

```text
session45/samples/looping_worker/
    Dockerfile
    worker.py
```

This sample keeps running and prints a message every 5 seconds.

Build:

```bash
cd session45/samples/looping_worker
docker build -t looping_worker .
```

Run in detached mode:

```bash
docker run -d --name my_worker looping_worker
```

View logs:

```bash
docker logs -f my_worker
```

Stop and remove:

```bash
docker stop my_worker
docker rm my_worker
```

Concept:

> A container stays running while its main process is running.


## Sample 5: Simple HTTP Server and Ports

Files:

```text
session45/samples/simple_http/
    Dockerfile
    server.py
```

This sample receives HTTP requests and sends a response.

Dockerfile:

```dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY server.py /app/

EXPOSE 8000

CMD ["python", "server.py"]
```

Build and run:

```bash
cd session45/samples/simple_http
docker build -t simple_http .
docker run -p 8000:8000 simple_http
```

Test from another terminal:

```bash
curl http://localhost:8000
```

Expected response:

```text
Hello from Docker container
```

Concept:

```text
localhost:8000 on your computer → port 8000 inside the container
```

This is why web apps need port mapping.


## Port Mapping

A container has its own network namespace. If an app listens inside the container, your host machine cannot automatically access it.

Use `-p`:

```bash
docker run -p HOST_PORT:CONTAINER_PORT image_name
```

Example:

```bash
docker run -p 8000:8000 simple_http
```

This means:

```text
localhost:8000 on host → port 8000 inside container
```

`EXPOSE` in Dockerfile documents the intended port, but `-p` actually publishes it.


## Installing Dependencies

The samples above use only the Python standard library.

If a Python project has dependencies, put them in `requirements.txt`:

```text
requests==2.32.0
```

Then use this Dockerfile pattern:

```dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt /app/
RUN pip install --no-cache-dir -r requirements.txt

COPY . /app/

CMD ["python", "main.py"]
```

Why copy `requirements.txt` first?

Docker can reuse the dependency installation step when only your Python code changes.


## Small Curious Note: Build Cache

Docker tries to reuse previous build steps when files have not changed.

That is why this order is useful:

```dockerfile
COPY requirements.txt /app/
RUN pip install --no-cache-dir -r requirements.txt
COPY . /app/
```

If only your Python code changes, Docker may reuse the dependency installation step.

You do not need to deeply understand image layers now. Just remember:

> Put dependency installation before copying all source code when possible.


## `.dockerignore`

`.dockerignore` tells Docker which files should not be copied into the build context.

Example:

```dockerignore
__pycache__/
*.pyc
*.pyo
.git/
.env
.env.*
venv/
.pytest_cache/
```

Why it matters:

- Smaller build context.
- Faster builds.
- Avoid leaking secrets.
- Avoid copying local virtual environments.


## CMD vs ENTRYPOINT

`CMD` gives the default command:

```dockerfile
CMD ["python", "greet.py"]
```

You can override it:

```bash
docker run greeting_app python greet.py Sara
```

`ENTRYPOINT` is more fixed:

```dockerfile
ENTRYPOINT ["python"]
CMD ["greet.py"]
```

For beginner app Dockerfiles, `CMD` is usually enough.


## Inspecting and Debugging Images

List images:

```bash
docker images
```

Run a shell inside an image:

```bash
docker run -it greeting_app sh
```

View logs of a running container:

```bash
docker logs CONTAINER_ID
```

For long-running containers, use:

```bash
docker logs -f CONTAINER_ID
```


## Optional: Tag and Publish Later

For this course, building and running locally is enough:

```bash
docker build -t greeting_app .
docker run greeting_app
```

Later, in real teams, images can be pushed to Docker Hub or another registry:

```bash
docker tag greeting_app username/greeting_app:latest
docker push username/greeting_app:latest
```

This is useful for CI/CD and deployment, but it is optional for now.


## Common Beginner Mistakes

| Mistake | Result | Fix |
|---------|--------|-----|
| Forgetting to copy a needed file | App cannot find the file | Add it with `COPY` |
| Expecting container filesystem to persist | Data disappears after removal | Use volumes or bind mounts |
| Forgetting `-p` for web apps | App runs but cannot be reached | Use `-p HOST:CONTAINER` |
| App listens on `127.0.0.1` inside container | Port mapping may not work correctly | Listen on `0.0.0.0` |
| Copying `.env` into image | Secrets may leak | Add `.env` to `.dockerignore` |
| Confusing image and container | Wrong command usage | Remember: image is template, container is instance |


## Summary

- A Dockerfile defines how to build an image.
- `RUN` executes during build; `CMD` executes when the container starts.
- Use `docker build` to create an image.
- Use `docker run` to create/start a container.
- Use `COPY` for files the container needs.
- Use bind mounts/volumes for data that should survive or be shared.
- Use detached mode and `docker logs -f` for long-running workers.
- Use `EXPOSE` and `-p` for request/response apps.
- Use `.dockerignore` to keep builds small and secrets safe.
- Django/DRF Dockerfile examples are better placed in a later Django deployment lesson.
